## Load EEG and preprocesses in same fashion as Alice dataset

Loads EEG data from data_preprocessed but adds preprocessing steps to align with preprocessing of Alice dataset.

In [5]:
import sys
# Add parent directory to path to import file_paths module
sys.path.insert(0, '..')
import helper_functions
from constants import *

import eelbrain
import mne
import numpy as np
import scipy.io


SUBJECTS = helper_functions.get_subjects()

In [6]:
def preprocess_eeg(info, subject, padded=False):
    """
    Preprocess and save EEG data for a single subject.

    Args:
        subject: Subject ID.
        padded:  Whether to also save a padded version.
    """
    dst_dir = EEG_DIR / subject
    dst_dir.mkdir(exist_ok=True, parents=True)

    suffix   = "_padded" if padded else ""
    eeg_path = dst_dir / f"{subject}_eeg{suffix}.pickle"

    if eeg_path.exists():
        print(f"{subject}: EEG exists, skipping.")
        return

    print(f"{subject}: preprocessing EEG...")

    mat  = scipy.io.loadmat(DATA_PREPROC / f"{subject}_data_preproc.mat", squeeze_me=True, struct_as_record=False)
    data = mat["data"]

    trials = []

    for trial in data.eeg:
        eeg = np.array(trial).T
        raw = mne.io.RawArray(eeg * 1e-6, info, verbose=False)

        raw.set_eeg_reference(['EXG1', 'EXG2'])
        raw.filter(EEG_BANDPASS_FILTER_LOW, EEG_BANDPASS_FILTER_HIGH, verbose=False)

        eeg_ndvar = eelbrain.load.mne.raw_ndvar(raw)

        if padded:
            eeg_ndvar = eelbrain.pad(eeg_ndvar, tstart=-PADDING_ONSET, tstop=eeg_ndvar.time.tstop + PADDING_OFFSET)

        trials.append(eeg_ndvar)

    eeg_concat = eelbrain.concatenate(trials, name=f"{subject}_eeg{suffix}")
    eelbrain.save.pickle(eeg_concat, eeg_path)

    print(f"  ✓ Saved {eeg_path}")

In [7]:
# MNE setup
montage   = mne.channels.make_standard_montage('biosemi64')
ch_names  = montage.ch_names + ['EXG1', 'EXG2']
ch_types  = ['eeg'] * 64 + ['misc', 'misc']
info      = mne.create_info(ch_names, EEG_SAMPLING_RATE, ch_types)
info.set_montage(montage, on_missing='ignore')

for subject in SUBJECTS:
    preprocess_eeg(info, subject, padded=False)
    preprocess_eeg(info, subject, padded=True)

print("Done preprocessing EEG.")

S1: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S1/S1_eeg.pickle
S1: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S1/S1_eeg_padded.pickle
S2: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S2/S2_eeg.pickle
S2: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S2/S2_eeg_padded.pickle
S3: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S3/S3_eeg.pickle
S3: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S3/S3_eeg_padded.pickle
S4: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S4/S4_eeg.pickle
S4: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S4/S4_eeg_padded.pickle
S5: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S5/S5_eeg.pickle
S5: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/cocoha4/eeg/S5/S5_eeg_padded.pickle
S6: preprocessing EEG...
  ✓ Saved /Users/sylvestereley/Data/coco

In [8]:
# SANITY CHECK: Check dimensions of preprocessed EEG
subject = SUBJECTS[0]
for padded in [False, True]:
    suffix   = "_padded" if padded else ""
    eeg_path = EEG_DIR / subject / f"{subject}_eeg{suffix}.pickle"
    if eeg_path.exists():
        print(f"  ✓ eeg{suffix}: {eelbrain.load.unpickle(eeg_path)}")
    else:
        print(f"  ✗ eeg{suffix}: MISSING")

  ✓ eeg: <NDVar 'S1_eeg': 64 sensor, 192000 time>
  ✓ eeg_padded: <NDVar 'S1_eeg_padded': 64 sensor, 196260 time>
